# Calculate extreme VPD threshold 

In [67]:
from datetime import timedelta
import xarray as xr
import numpy as np
import pandas as pd

Define start and end date to calcualte extreme VPD thresholds

In [70]:
start_date = '1980-01-01'
end_date = '2014-12-31'

Function to calculate VPD threshold for CNRM-CERFACS_CNRM-CM6-1-HR, MOHC_HadGEM3-GC31-MM, and NOAA-GFDL_GFDL-ESM4

In [ ]:
def calculate_VPD_threshold(model, ssp):    
    his_file_path = '..\\VPD\\' + model + '\\historical\\historical_' + model + '_VPD_filtered.csv'
    all_his = pd.read_csv(his_file_path, parse_dates=['time'])
    # print(all_his)
    
    ssp_file_path = '..\\VPD\\' + model + '\\' + ssp + '\\' + ssp + '_' + model + '_VPD_filtered.csv'
    all_ssp = pd.read_csv(ssp_file_path, parse_dates=['time'])
    # print(all_ssp)
    
    all_data = pd.concat([all_his, all_ssp], axis=0) #concat historical data and ssp data

    #hourly to daily max
    all_data_dailymax = all_data.resample('D',on='time').max()
    # print(all_his_dailymax)
    
    # Ensure the index is a datetime type
    all_data_dailymax.index = pd.to_datetime(all_data_dailymax.index)
    # Filter data from 2001-01-01 to 2015-12-31
    filtereds_dailymax = all_data_dailymax.loc[start_date: end_date]
    # print(filtered_his_dailymax)
    
    # Calculate threshold
    threshold = filtereds_dailymax['VPD'].quantile(0.984)
    # print(threshold)
    return threshold

Calculate threshold for CNRM-CERFACS_CNRM-CM6-1-HR, MOHC_HadGEM3-GC31-MM, and NOAA-GFDL_GFDL-ESM4

In [76]:
model = 'CNRM-CERFACS_CNRM-CM6-1-HR'
CNRM_threshold = calculate_VPD_threshold(model, 'ssp126')
print(f'CNRM-CERFACS_CNRM-CM6-1-HR {CNRM_threshold}')

model = 'MOHC_HadGEM3-GC31-MM'
MOHC_threshold = calculate_VPD_threshold(model, 'ssp126')
print(f'MOHC_HadGEM3-GC31-MM {MOHC_threshold}')

model = 'NOAA-GFDL_GFDL-ESM4'
NOAA_threshold = calculate_VPD_threshold(model, 'ssp126')
print(f'NOAA-GFDL_GFDL-ESM4 {NOAA_threshold}')

CNRM-CERFACS_CNRM-CM6-1-HR 27.024601353296315
MOHC_HadGEM3-GC31-MM 38.4032999247096
NOAA-GFDL_GFDL-ESM4 51.17654503990564


Function to calculate VPD threshold for CESM

In [ ]:
def calculate_VPD_threshold_CESM(model, ssp):    
    his_file_path = '..\\VPD\\' + model + '\\historical\\historical_' + model + '_VPD.csv'
    all_his = pd.read_csv(his_file_path, parse_dates=['date'])
    all_his.rename(columns={'CESM_hist_vpd': 'CESM_VPD'}, inplace=True)
    
    ssp_file_path = '..\\VPD\\' + model + '\\SSP\\' + model + '_ssp_combined.csv'
    all_ssp = pd.read_csv(ssp_file_path, parse_dates=['date'])
    all_ssp.rename(columns={ssp + '_VPD': 'CESM_VPD'}, inplace=True)
    # print(all_ssp)

    all_data = pd.concat([all_his, all_ssp], axis=0) #concat historical data and ssp data
    # print(all_data)
    # all_data.to_csv('all_data.csv')
#     # print(all_his)
    all_data['date'] = all_data['date'] -timedelta(hours=4) #convert UTC to local time
    # print(all_his)
    #hourly to daily max
    all_data_dailymax = all_data.resample('D',on='date').max().loc[start_date: end_date]
    # print(all_his_dailymax)
    
    # Calculate threshold
    threshold = all_data_dailymax['CESM_VPD'].quantile(0.984)
    # print(threshold)
    return threshold

Calculate threshold for CESM

In [83]:
model = 'CESM'
CNRM_threshold = calculate_VPD_threshold_CESM(model, 'SSP126') #SSP should be uppercase
print(f'CNRM_threshold {CNRM_threshold}')

CNRM_threshold 44.21403417992513


Function to calculate VPD threshold for E3SM_BCRD and E3SM_BDRD

In [ ]:
def calculate_VPD_threshold_E3SM(model):    
    file_path = '..\\VPD\\' + model + '\\' + model + '_VPD.csv'
    all_data = pd.read_csv(file_path, parse_dates=['time_bnds'])
    # print(all_his)
    
    all_data['time_bnds'] = all_data['time_bnds'] -timedelta(hours=4) #convert UTC to local time

    # print(all_his)
    # all_his.to_csv('all_his.csv')
    #hourly to daily max
    all_data_dailymax = all_data.resample('D',on='time_bnds').max()
    # print(all_data_dailymax)
    # all_his_dailymax.to_csv('all_his_dailymax.csv')

    # Filter data from 2001-01-01 to 2015-12-31
    filtered_dailymax = all_data_dailymax.loc[start_date: end_date]
    # print(filtered_dailymax)
    
    # Calculate threshold
    threshold = filtered_dailymax['vpd'].quantile(0.984)
    # print(threshold)
    return threshold

Calculate threshold for E3SM_BCRD and E3SM_BDRD

In [90]:
model = 'E3SM_BCRD'
E3SM_BCRD_threshold = calculate_VPD_threshold_E3SM(model)
print(f'E3SM_BCRD_threshold {E3SM_BCRD_threshold}')

model = 'E3SM_BDRD'
E3SM_BDRD_threshold = calculate_VPD_threshold_E3SM(model)
print(f'E3SM_BDRD_threshold {E3SM_BDRD_threshold}')

E3SM_BCRD_threshold 36.96984881497566
E3SM_BDRD_threshold 35.48145103898043
